In [1]:
import os

import pandas as pd

from dataset_util import dataframes2objects, create_yolo_dataset

%load_ext autoreload
%autoreload 2

/home/klymy/.pyenv/versions/3.10.13/envs/pgpytrain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# base_path = "../../../db/object-detection"
base_path = "/mnt/pg-fs01/Current/DB/ObjectDetectionV3"
# base_path = "../../../data/ObjectDetectionV3"
# base_path = "database"
df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df.head()

,label,occluded,truncated,not_ground,occupied,occupied_by_person,ID,xtl,ytl,xbr,ybr,x,y,img_id,id
0,meeting_room_table,True,False,False,False,False,8.0,51.60,186.10,307.89,332.61,172.16,260.70,bdd640fb-0667-4ad1-9c80-317fa3b1799d,23b8c1e9-3924-46de-beb1-3b9046685257
1,bag,False,False,False,False,False,9.0,346.45,165.42,393.28,220.25,369.91,191.48,bdd640fb-0667-4ad1-9c80-317fa3b1799d,bd9c66b3-ad3c-4d6d-9a3d-1fa7bc8960a9
2,bag,True,False,False,False,False,10.0,329.10,133.89,369.63,170.60,350.15,150.37,bdd640fb-0667-4ad1-9c80-317fa3b1799d,972a8469-1641-4f82-8b9d-2434e465e150
3,laptop_closed,False,False,False,False,False,19.0,224.38,287.20,280.56,324.61,253.60,305.88,bdd640fb-0667-4ad1-9c80-317fa3b1799d,17fc695a-07a0-4a6e-8822-e8f36c031199
4,laptop_opened,False,False,False,False,False,18.0,197.42,191.27,249.80,228.00,224.38,210.00,bdd640fb-0667-4ad1-9c80-317fa3b1799d,9a1de644-815e-46d1-bb8f-aa1837f8a88b


In [14]:
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))
df_imgs.head()

,id,name,width,height,batch,folder,group_id
0,bdd640fb-0667-4ad1-9c80-317fa3b1799d,Cam3_H243_discount_14.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
1,fc377a4c-4a15-444d-85e7-ce8a3a578a8e,Cam3_H243_discount_15.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
2,ff50bde4-3825-47b8-9cab-cc97663f1c97,Cam3_H243_discount_16.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
3,5cec4eb5-edd9-4831-9ca3-5cfb04fc6d82,Cam3_H243_discount_17.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b
4,b92da22b-21df-406f-8a0b-3c3336d8393a,Cam3_H243_discount_18.png,640.0,480.0,batch1_10.5,batch1,120287d6-9abf-49b8-ac00-72c72f9e383b


In [18]:
# filter objects:
# - remove occupied by person
# - remove objects that occures in dataset < 1000 times
dataset_name = "baseline"

df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

df = df[df["occupied_by_person"] == "False"]

popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
office_chair     5267
desk             3384
desk_FULL        2294
laptop_opened    2049
monitor          1899
Name: count, dtype: int64
740 165


train labels: 740it [00:00, 5538.92it/s]
val labels: 165it [00:00, 9654.45it/s]
train images: 740it [00:05, 129.45it/s]
val images: 165it [00:01, 130.19it/s]


In [19]:
# filter objects:
# - remove objects that occures in dataset < 1000 times
dataset_name = "with_occupied_objects"

df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
office_chair     7867
desk             3385
desk_FULL        2300
laptop_opened    2060
monitor          1900
Name: count, dtype: int64
740 165


train labels: 740it [00:00, 13939.70it/s]
val labels: 165it [00:00, 10498.96it/s]
train images: 740it [00:05, 125.61it/s]
val images: 165it [00:01, 133.47it/s]


In [20]:
# filter objects:
# - rm occupied by person chairs
# - remove objects that occures in dataset < 1000 times
dataset_name = "no_occupied_chairs"

df = pd.read_csv(os.path.join(base_path, "annotation.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

df = df[~((df["label"] == "office_chair") & (df["occupied_by_person"] == "True"))]

popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
office_chair     5267
desk             3385
desk_FULL        2300
laptop_opened    2060
monitor          1900
Name: count, dtype: int64
740 165


train labels: 740it [00:00, 7759.79it/s]
val labels: 165it [00:00, 10217.32it/s]
train images: 740it [00:05, 130.33it/s]
val images: 165it [00:01, 127.92it/s]


In [7]:
# filter objects:
# - remove occupied by person
# - remove objects that occures in dataset < 1000 times
dataset_name = "merged_furniture_no_occupied"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

df = df[df["occupied_by_person"] != "True"]

popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path,
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name),
    os.path.join(base_path, "train_test_split.json"),
)

label
chair            8977
table            6002
laptop_opened    3009
monitor          2881
keyboard         1419
plants           1408
cabinet          1112
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 5151.56it/s]
val labels: 292it [00:00, 6805.04it/s]
train images: 1027it [00:00, 4178.26it/s]
val images: 292it [00:00, 3948.28it/s]


In [21]:
# filter objects:
# - remove objects that occures in dataset < 1000 times
dataset_name = "merged_furniture"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair            9069
table            4130
laptop_opened    2060
monitor          1900
Name: count, dtype: int64
740 165


train labels: 740it [00:00, 5244.57it/s]
val labels: 165it [00:00, 8128.88it/s]
train images: 740it [00:05, 130.71it/s]
val images: 165it [00:01, 138.29it/s]


In [23]:
# filter objects:
# - rm occupied by person chairs
# - remove objects that occures in dataset < 1000 times
dataset_name = "merged_furniture_no_occupied_chairs"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

df = df[~((df["label"] == "chair") & (df["occupied_by_person"] == "True"))]

popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair            6409
table            4130
laptop_opened    2060
monitor          1900
Name: count, dtype: int64
740 165


train labels: 740it [00:00, 4126.02it/s]
val labels: 165it [00:00, 8805.62it/s]
train images: 740it [00:06, 123.26it/s]
val images: 165it [00:01, 126.91it/s]


In [3]:
# dataset v2
base_path = "../../../data/ObjectDetectionV3"

In [26]:
# filter objects:
# - remove objects that occures in dataset < 1000 times
dataset_name = "merged_furniture"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair            13518
table             6011
laptop_opened     3020
monitor           2882
keyboard          1422
plants            1408
cabinet           1119
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 4494.19it/s]
val labels: 292it [00:00, 5279.47it/s]
train images: 1027it [00:00, 3991.02it/s]
val images: 292it [00:00, 4021.62it/s]


In [27]:
# filter objects:
# - rm occupied by person chairs
# - remove objects that occures in dataset < 1000 times
dataset_name = "merged_furniture_no_occupied_chairs"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

df = df[~((df["label"] == "chair") & (df["occupied_by_person"] == "True"))]

popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair            8977
table            6011
laptop_opened    3020
monitor          2882
keyboard         1422
plants           1408
cabinet          1119
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 13521.69it/s]
val labels: 292it [00:00, 6769.38it/s]
train images: 1027it [00:00, 4256.47it/s]
val images: 292it [00:00, 4201.22it/s]


In [5]:
# filter objects:
# - take classes that accures in prev version of dataset
dataset_name = "merged_furniture_4_classes"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

# popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
popular_labels = ["chair", "table", "laptop_opened", "monitor"]
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair            13518
table             6011
laptop_opened     3020
monitor           2882
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 5023.39it/s]
val labels: 292it [00:00, 6388.89it/s]
train images: 1027it [00:00, 4204.29it/s]
val images: 292it [00:00, 4166.08it/s]


In [6]:
# filter objects:
# - rm occupied by person chairs
# - take classes that accures in prev version of dataset
dataset_name = "merged_furniture_no_occupied_chairs_4_classes"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

df = df[~((df["label"] == "chair") & (df["occupied_by_person"] == "True"))]

# popular_labels = df["label"].value_counts()[df["label"].value_counts() >= 1000].index.values
popular_labels = ["chair", "table", "laptop_opened", "monitor"]
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair            8977
table            6011
laptop_opened    3020
monitor          2882
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 5678.36it/s]
val labels: 292it [00:00, 7410.74it/s]
train images: 1027it [00:00, 4237.40it/s]
val images: 292it [00:00, 3992.80it/s]


In [12]:
# add new class - chair_occupied

In [17]:
dataset_name = "merged_furniture_chair_occupied"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

condition = (df["label"] == "chair") & (df["occupied_by_person"] == "True")
df.loc[condition, "label"] = "chair_occupied"

popular_labels = ["chair", "table", "laptop_opened", "monitor", "chair_occupied", "keyboard", "plants", "cabinet"]
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs, labels=popular_labels)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair             8977
table             6011
chair_occupied    4541
laptop_opened     3020
monitor           2882
keyboard          1422
plants            1408
cabinet           1119
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 4253.75it/s]
val labels: 292it [00:00, 5755.45it/s]
train images: 1027it [00:00, 1035.97it/s]
val images: 292it [00:00, 755.66it/s]


In [18]:
dataset_name = "merged_furniture_chair_occupied_5_classes"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

condition = (df["label"] == "chair") & (df["occupied_by_person"] == "True")
df.loc[condition, "label"] = "chair_occupied"

popular_labels = ["chair", "table", "laptop_opened", "monitor", "chair_occupied"]
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs, labels=popular_labels)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair             8977
table             6011
chair_occupied    4541
laptop_opened     3020
monitor           2882
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 5124.14it/s]
val labels: 292it [00:00, 6850.33it/s]
train images: 1027it [00:00, 1180.12it/s]
val images: 292it [00:00, 1076.98it/s]


In [5]:
# only two classes
dataset_name = "merged_furniture_table_non_ocupied_chair"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

# condition = (df["label"] == "chair") & (df["occupied_by_person"] == "True")
# df.loc[condition, "label"] = "chair_occupied"
df = df[~((df["label"] == "chair") & (df["occupied_by_person"] == "True"))]

popular_labels = ["chair", "table"]
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs, labels=popular_labels)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair    8977
table    6011
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 13606.56it/s]
val labels: 292it [00:00, 8188.66it/s]
train images: 1027it [00:00, 3976.20it/s]
val images: 292it [00:00, 3902.18it/s]


In [6]:
# only two classes
dataset_name = "merged_furniture_table_chair"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

# condition = (df["label"] == "chair") & (df["occupied_by_person"] == "True")
# df.loc[condition, "label"] = "chair_occupied"
# df = df[~((df["label"] == "chair") & (df["occupied_by_person"] == "True"))]

popular_labels = ["chair", "table"]
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs, labels=popular_labels)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair    13518
table     6011
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 5820.42it/s]
val labels: 292it [00:00, 7730.56it/s]
train images: 1027it [00:00, 3762.29it/s]
val images: 292it [00:00, 2848.14it/s]


In [7]:
# only two classes
dataset_name = "merged_furniture_table_chair_chair_occupied"

df = pd.read_csv(os.path.join(base_path, "annotation_merged_furniture.csv"))
df_imgs = pd.read_csv(os.path.join(base_path, "images_grouped.csv"))

condition = (df["label"] == "chair") & (df["occupied_by_person"] == "True")
df.loc[condition, "label"] = "chair_occupied"
# df = df[~((df["label"] == "chair") & (df["occupied_by_person"] == "True"))]

popular_labels = ["chair", "table", "chair_occupied"]
df = df[df["label"].isin(popular_labels)]

print(df["label"].value_counts())

objects, label_mapping = dataframes2objects(df, df_imgs, labels=popular_labels)
create_yolo_dataset(
    objects, label_mapping, base_path, 
    os.path.join(os.getcwd(), "datasets", "v2_batch6", dataset_name), 
    os.path.join(base_path, "train_test_split.json")
)

label
chair             8977
table             6011
chair_occupied    4541
Name: count, dtype: int64
1027 292


train labels: 1027it [00:00, 5552.71it/s]
val labels: 292it [00:00, 8183.46it/s]
train images: 1027it [00:00, 3988.08it/s]
val images: 292it [00:00, 3619.89it/s]
